# Baseline classifier: training, testing, running

Fine-tunes an Ultralytics YOLO classification model (`yolo26n-cls.pt`) on
SID-Set to tell real photos from AI-generated/tampered ones.

This notebook drives the pipeline entirely through the shared classes in
`packages/models/normal_classifier`, instead of ad-hoc inline code:

- **Training + testing** — `NormalClassifierTrainer`, which extends
  `shared_types.TrainableModel` (`.train()` / `.evaluate()` / `.save()` / `.load()`).
- **Running (inference)** — `NormalClassifierDetector`, which implements
  `shared_types.EnsembleDetector` (`.predict()`) — the same "ready" contract
  `apps/web`'s Streamlit demo already knows how to consume.

Data comes from `data.dataset_builder`, which streams SID-Set from Hugging
Face rather than downloading it: `iter_sid_subset()` for the large,
single-use training pull (never holds more than one decoded image in
memory at a time), and `load_sid_subset()` + `to_labeled_samples()` for
the small validation pull that gets reused across cells below. Both
produce the shared `LabeledImageSample` type the classes above expect.


## 1. Setup — clone the repo, install deps, wire up imports

In [ ]:
# Install uv, then use it to install this monorepo's local packages
# straight into Colab's own active Python -- no separate venv, no editable
# ".pth" redirects that would need extra `site` processing to become
# importable in an already-running kernel. To work on a different model
# package later, just change PACKAGE below (it must only depend on
# shared_types/image_io/data, like the others under packages/models/).
%cd /content/TikTokTechJam2026
!git switch "setup"
!git pull

%pip install -q uv

PACKAGE = "normal_classifier"

import importlib
import importlib.metadata
import subprocess
import sys


def _installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


numpy_before = _installed_version("numpy")

# Our own local packages, --no-deps: Colab's base image already has numpy/
# pillow satisfying what these declare, so there's no need to let uv
# re-resolve them here. Letting it do so risks upgrading/replacing an
# already-imported numpy on disk while the OLD version is still loaded in
# this running kernel's memory -- Python never reloads an already-imported
# C extension, so you get a pure-Python-vs-compiled-binary mismatch (the
# "no attribute '_blas_supports_fpe'" class of AttributeError).
subprocess.run(
    [
        "uv", "pip", "install", "--system", "--python", sys.executable, "--no-deps",
        "./libs/shared_types",
        "./libs/image_io",
        "./packages/data",
        f"./packages/models/{PACKAGE}",
    ],
    check=True,
)

# The genuinely-missing third-party packages these need (not already in
# Colab's base image), installed separately so their own dependency
# resolution doesn't get tangled up with the --no-deps installs above.
subprocess.run(
    ["uv", "pip", "install", "--system", "--python", sys.executable, "ultralytics>=8.3", "datasets>=2.19"],
    check=True,
)

numpy_after = _installed_version("numpy")
if numpy_before is not None and numpy_before != numpy_after:
    print(
        f"
*** numpy changed ({numpy_before} -> {numpy_after}) while this kernel was already "
        "running with the old version loaded in memory. RESTART THE RUNTIME NOW "
        "(Runtime > Restart session), then re-run this notebook from the top -- "
        "otherwise later cells will hit numpy AttributeErrors. ***
"
    )

# Regular installs land as real files in site-packages, so nothing further
# is needed for imports to work -- this just forces Python to notice files
# that appeared after the interpreter already started.
importlib.invalidate_caches()

importlib.import_module(PACKAGE)
print(f"'{PACKAGE}' imported OK")


## 2. Load training + validation data from SID-Set

In [ ]:
from data.dataset_builder import iter_sid_subset, load_sid_subset, to_labeled_samples

# Training pull is large (1000/label) and only used once, so stream it
# straight through -- iter_sid_subset() never holds more than one decoded
# image in memory at a time, unlike materializing the whole set up front.
train_samples = iter_sid_subset(images_per_label=1000, split="train")

# Validation pull is small and reused three times below (train's val_samples,
# evaluate(), and the final inference demo), so it's fine to materialize.
val_images, val_metadata = load_sid_subset(images_per_label=200, split="validation")
val_samples = to_labeled_samples(val_images, val_metadata)

print(f"val samples: {len(val_samples)} (train_samples is a generator -- consumed by trainer.train() below)")


## 3. Train

`NormalClassifierTrainer.train()` exports `train_samples`/`val_samples` into
the `real/`, `ai_generated/` class-folder layout Ultralytics' classification
trainer expects, then fine-tunes `yolo26n-cls.pt` on them.

In [ ]:
from normal_classifier import NormalClassifierTrainer

trainer = NormalClassifierTrainer(base_weights="yolo26n-cls.pt", image_size=224)
result = trainer.train(
    train_samples,
    val_samples=val_samples,
    output_dir="SID_YOLO",
    epochs=100,
    batch=32,
    patience=10,
    device="cpu",
    plots=True,
)
result


## 4. Test

The "testing" stage — score the trained model against a held-out set via
`.evaluate()`. SID-Set only exposes train/validation splits, so this reuses
`val_samples`; swap in a separate held-out set here if you have one.

In [ ]:
metrics = trainer.evaluate(val_samples, output_dir="SID_YOLO_eval")
print("Held-out evaluation metrics:", metrics)


In [ ]:
trainer.save("normal_classifier.pt")
print("Saved checkpoint to normal_classifier.pt")


## 5. Run (inference)

The "running" stage — `NormalClassifierDetector` wraps the saved checkpoint
and implements the same `EnsembleDetector` contract `apps/web` consumes, so
this class can be dropped straight into
`apps/web/src/web/services/factory.py`'s `get_detector()` once ready.

In [ ]:
from normal_classifier import NormalClassifierDetector

detector = NormalClassifierDetector.from_checkpoint("normal_classifier.pt")

for sample in val_samples[:5]:
    detection = detector.predict(sample.image)
    print(
        f"true={sample.metadata['label_name']:<10} "
        f"predicted={detection.verdict:<12} "
        f"p(ai_generated)={detection.ai_generated_probability:.2f}"
    )
